# **LightGBM**

## **1. Import Libraries**

In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from data_preparation import data_preparation

## **2. Data Preparation**

In [55]:
X_train, X_test, y_train, y_test = data_preparation(
    '../dataset/digital_marketing_campaign_dataset.csv'
)

## **3. Preprocessing**

In [56]:
X_train['CampaignType'] = X_train['CampaignType'].astype('category')
X_test['CampaignType']  = X_test['CampaignType'].astype('category')

## **4. Validation Split**

Split a held-out validation set from the training data so early stopping does **not** peek at test data (prevents data leakage).

In [57]:
from sklearn.model_selection import train_test_split

X_train2, X_val, y_train2, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

## **5. Hyperparameter Configuration**

In [58]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 96,
    'max_depth': 7,
    'learning_rate': 0.005,
    'feature_fraction': 0.82,
    'bagging_fraction': 0.85,
    'bagging_freq': 5,
    'min_child_samples': 35,
    'scale_pos_weight': len(y_train2[y_train2 == 0]) / len(y_train2[y_train2 == 1]),
    'lambda_l1': 1.5,
    'lambda_l2': 2.0,
    'verbose': -1,
    'num_threads': -1
}

train_set = lgb.Dataset(X_train2, label=y_train2)
valid_set = lgb.Dataset(X_val,    label=y_val,    reference=train_set)

## **6. Training**

In [59]:
evals_result = {}

model = lgb.train(
    params,
    train_set,
    num_boost_round=4000,
    valid_sets=[train_set, valid_set],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=150, verbose=True),
        lgb.log_evaluation(100),
        lgb.record_evaluation(evals_result)
    ]
)

Training until validation scores don't improve for 150 rounds
[100]	train's auc: 0.853487	valid's auc: 0.819667
[200]	train's auc: 0.874591	valid's auc: 0.8355
[300]	train's auc: 0.890561	valid's auc: 0.844423
[400]	train's auc: 0.90166	valid's auc: 0.84785
[500]	train's auc: 0.910802	valid's auc: 0.849839
[600]	train's auc: 0.918898	valid's auc: 0.849359
Early stopping, best iteration is:
[495]	train's auc: 0.910459	valid's auc: 0.849979


## **11. Save Model**

Using LightGBM's native format for better cross-version compatibility.

In [65]:
model.save_model('../models/lgbm_model.txt')